# BL-1 Quickstart: Simulating a Cortical Culture

This notebook walks through the core workflow of the BL-1 simulator in five minutes:

1. **Create** a small network of Izhikevich spiking neurons (80/20 E/I split)
2. **Connect** them with distance-dependent synapses
3. **Simulate** 1 second of spontaneous + driven activity
4. **Visualize** the spike raster and population firing rate
5. **Record** from a virtual 64-channel MEA

## 1. Imports and Setup

In [ ]:
import jax
import jax.numpy as jnp
import matplotlib.pyplot as plt
import numpy as np

from bl1.core.izhikevich import create_population, IzhikevichParams, NeuronState
from bl1.core.synapses import create_synapse_state, SynapseState
from bl1.core.integrator import simulate
from bl1.network.topology import place_neurons, build_connectivity
from bl1.mea.electrode import MEA, build_neuron_electrode_map
from bl1.mea.recording import detect_spikes, compute_electrode_rates
from bl1.visualization.raster import plot_raster, plot_raster_with_rate

# Reproducibility
key = jax.random.PRNGKey(42)
print("JAX backend:", jax.default_backend())

## 2. Create the Network

We build a 200-neuron culture on a 3000 x 3000 um substrate (matching a
standard MEA dish). The population has an 80/20 excitatory/inhibitory split
with five Izhikevich cell types: RS, IB, CH (excitatory) and FS, LTS
(inhibitory).

In [ ]:
N_NEURONS = 200
DT = 0.5  # ms per timestep (Izhikevich recommends 0.5 ms half-steps)

k1, k2, k3 = jax.random.split(key, 3)

# 2a. Place neurons uniformly on the substrate
positions = place_neurons(k1, N_NEURONS, substrate_um=(3000.0, 3000.0))

# 2b. Create the neuron population (parameters + initial state)
izh_params, neuron_state, is_excitatory = create_population(k2, N_NEURONS, ei_ratio=0.8)

n_exc = int(is_excitatory.sum())
n_inh = N_NEURONS - n_exc
print(f"Neurons: {N_NEURONS} total ({n_exc} excitatory, {n_inh} inhibitory)")
print(f"Positions shape: {positions.shape}")

# 2c. Build distance-dependent connectivity (returns BCOO sparse matrices)
W_exc, W_inh, delays = build_connectivity(
    k3, positions, is_excitatory,
    lambda_um=200.0,   # length constant for connection probability
    p_max=0.21,        # max connection probability at zero distance
    g_exc=0.05,        # excitatory weight
    g_inh=0.20,        # inhibitory weight (4x stronger, as in cortex)
    dt=DT,
)

print(f"Excitatory synapses: {W_exc.nse}")
print(f"Inhibitory synapses: {W_inh.nse}")

In [ ]:
# Quick plot: neuron positions coloured by E/I type
fig, ax = plt.subplots(figsize=(6, 6))
pos_np = np.asarray(positions)
exc = np.asarray(is_excitatory)

ax.scatter(pos_np[exc, 0], pos_np[exc, 1], s=8, c="steelblue", alpha=0.7, label="Excitatory")
ax.scatter(pos_np[~exc, 0], pos_np[~exc, 1], s=8, c="tomato", alpha=0.7, label="Inhibitory")
ax.set_xlabel("x (um)")
ax.set_ylabel("y (um)")
ax.set_title("Neuron Positions on Substrate")
ax.legend()
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

## 3. Run the Simulation

The integrator uses `jax.lax.scan` to compile the entire time loop into a
single XLA program -- no Python overhead per timestep.

We simulate 1 second (2000 timesteps at dt=0.5 ms) with a constant
background current to drive spontaneous activity. The four synapse types
(AMPA, NMDA, GABA_A, GABA_B) are updated automatically inside the
integrator.

In [ ]:
SIM_DURATION_MS = 1000.0
T_STEPS = int(SIM_DURATION_MS / DT)  # 2000 steps

# Initialize synapse state (all conductances start at zero)
syn_state = create_synapse_state(N_NEURONS)

# External drive: tonic current of 7 pA to all neurons.
# This puts RS neurons just above rheobase so the network
# generates spontaneous activity driven by recurrent synapses.
I_external = jnp.full((T_STEPS, N_NEURONS), 7.0)

# Run the simulation (first call includes JIT compilation time)
print(f"Simulating {SIM_DURATION_MS:.0f} ms ({T_STEPS} steps) ...")
result = simulate(
    params=izh_params,
    init_state=neuron_state,
    syn_state=syn_state,
    stdp_state=None,          # no plasticity for this demo
    W_exc=W_exc,
    W_inh=W_inh,
    I_external=I_external,
    dt=DT,
    plasticity_fn=None,
)

spike_history = result.spike_history  # (T, N) boolean array
total_spikes = int(jnp.sum(spike_history))
mean_rate_hz = total_spikes / N_NEURONS / (SIM_DURATION_MS / 1000.0)
print(f"Total spikes: {total_spikes}")
print(f"Mean firing rate: {mean_rate_hz:.1f} Hz")

## 4. Visualization

### 4a. Spike Raster

Each dot is one spike. Excitatory neurons (blue) are indexed below the
E/I boundary; inhibitory neurons (red) above it.

In [ ]:
fig = plot_raster(
    np.asarray(spike_history),
    dt_ms=DT,
    ei_boundary=n_exc,
    title="Spike Raster (1 s simulation)",
)
plt.show()

### 4b. Raster + Population Firing Rate

The bottom panel shows the population-average firing rate in 10 ms bins --
useful for spotting network bursts.

In [ ]:
fig = plot_raster_with_rate(
    np.asarray(spike_history),
    dt_ms=DT,
    rate_bin_ms=10.0,
    title="Neural Activity with Population Rate",
)
plt.show()

### 4c. Per-neuron Firing Rates

In [ ]:
# Per-neuron spike counts and firing rates
spike_counts = np.asarray(spike_history.sum(axis=0))
rates_hz = spike_counts / (SIM_DURATION_MS / 1000.0)

fig, ax = plt.subplots(figsize=(10, 3))
colors = np.where(np.asarray(is_excitatory), "steelblue", "tomato")
ax.bar(range(N_NEURONS), rates_hz, color=colors, width=1.0)
ax.set_xlabel("Neuron index")
ax.set_ylabel("Firing rate (Hz)")
ax.set_title("Per-neuron Firing Rates")
ax.axvline(n_exc - 0.5, color="gray", linestyle="--", linewidth=0.8, label="E/I boundary")
ax.legend()
plt.tight_layout()
plt.show()

print(f"Excitatory mean rate: {rates_hz[:n_exc].mean():.1f} Hz")
print(f"Inhibitory mean rate: {rates_hz[n_exc:].mean():.1f} Hz")

## 5. Virtual MEA Recording

BL-1 includes a virtual multi-electrode array that maps neuron spikes to
electrode channels based on spatial proximity. The default `cl1_64ch`
configuration is an 8x8 grid with 200 um spacing, matching the CorticalLabs
CL1 hardware.

In [ ]:
# Create the MEA and build the spatial mapping
mea = MEA("cl1_64ch")
print(f"MEA: {mea.config.name}, {mea.n_electrodes} electrodes")
print(f"Grid: {mea.config.grid_shape}, spacing: {mea.config.spacing_um} um")
print(f"Detection radius: {mea.detection_radius_um} um")

# Build neuron-electrode map: (E, N) boolean mask
ne_map = build_neuron_electrode_map(
    positions, mea.positions, mea.detection_radius_um
)
neurons_per_electrode = np.asarray(ne_map.sum(axis=1))
print(f"\nNeurons detected per electrode: "
      f"min={neurons_per_electrode.min()}, "
      f"max={neurons_per_electrode.max()}, "
      f"mean={neurons_per_electrode.mean():.1f}")

In [ ]:
# Visualize electrode positions overlaid on neurons
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(pos_np[:, 0], pos_np[:, 1], s=5, c="lightgray", alpha=0.5, label="Neurons")

elec_np = np.asarray(mea.positions)
ax.scatter(elec_np[:, 0], elec_np[:, 1], s=40, c="black", marker="s", label="Electrodes")

# Draw detection radii for a few electrodes
for i in range(0, mea.n_electrodes, 16):
    circle = plt.Circle(
        (float(elec_np[i, 0]), float(elec_np[i, 1])),
        mea.detection_radius_um, fill=False, color="gray", linestyle="--", linewidth=0.5,
    )
    ax.add_patch(circle)

ax.set_xlabel("x (um)")
ax.set_ylabel("y (um)")
ax.set_title("MEA Electrode Layout")
ax.legend(loc="upper right")
ax.set_aspect("equal")
plt.tight_layout()
plt.show()

In [ ]:
# Compute per-electrode firing rates over the full simulation
electrode_rates = compute_electrode_rates(
    spike_history, ne_map, window_ms=SIM_DURATION_MS, dt=DT
)
electrode_rates_np = np.asarray(electrode_rates)

# Display as an 8x8 heatmap matching the MEA grid
rate_grid = electrode_rates_np.reshape(mea.config.grid_shape)

fig, ax = plt.subplots(figsize=(6, 5))
im = ax.imshow(rate_grid, cmap="hot", interpolation="nearest")
ax.set_title("Electrode Firing Rates (Hz)")
ax.set_xlabel("Column")
ax.set_ylabel("Row")
plt.colorbar(im, ax=ax, label="Mean rate (Hz)")
plt.tight_layout()
plt.show()

print(f"Electrode rate range: {electrode_rates_np.min():.1f} - {electrode_rates_np.max():.1f} Hz")

## Next Steps

- **Plasticity**: Add STDP to let synaptic weights evolve -- see `bl1.plasticity.stdp`
- **Closed-loop**: Connect the culture to a Pong game via `bl1.loop.controller.ClosedLoop`
- **Analysis**: Measure criticality (`bl1.analysis.criticality`) and detect bursts (`bl1.analysis.bursts`)
- **Scale up**: Use `Culture.create(key, n_neurons=100_000)` with `use_fast_sparse=True` or `use_event_driven=True` for large-scale simulations